# PyMCDC — Modified Condition/Decision Coverage
Install: `pip install pymcdc` | CLI: `python -m pymcdc <file>`

In [15]:
import pymcdc
print(dir(pymcdc))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__main__', '__name__', '__package__', '__path__', '__spec__', 'cli', 'decisao', 'mcdc_html_generator', 'transform_log', 'visit_dot', 'visit_log']


In [16]:
# Raw submodule attributes
import pkgutil, importlib

for finder, name, ispkg in pkgutil.walk_packages(pymcdc.__path__, pymcdc.__name__ + '.'):
    try:
        mod = importlib.import_module(name)
        print(name, '->', dir(mod))
    except Exception as e:
        print(name, '-> ERROR:', e)

pymcdc.__main__ -> ['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'main', 'show_help', 'sys']
pymcdc.cli -> ['Argumento', 'Decisao', 'GREEN', 'LogTransformer', 'LogVisitor', 'Path', 'RED', 'RESET', '__builtins__', '__cached__', '__doc__', '__file__', '__ignore', '__loader__', '__name__', '__package__', '__spec__', '__total', '_ensure_module_hierarchy', '_install_module', '_to_module_name', 'argparse', 'ast', 'bool_register', 'dic_of_traces', 'end_bool_register', 'generate_mcdc_html', 'green', 'html_all_conditions', 'importlib', 'init_bool_register', 'main', 'os', 'parse_args', 'pickle', 'pilha_de_execucao', 'print_all_conditions', 'red', 'run_ast_tests', 'shlex', 'sys', 'time', 'types', 'unittest', 'usage']
pymcdc.decisao -> ['Counter', 'Decisao', 'List', 'LogVisitor', 'Set', 'Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'lru_cache']
pymcdc.mcdc_html_generato

In [17]:
# Raw pymcdc CLI output on sample code
import subprocess, tempfile, os

sample = '''
def authorize(is_admin, is_owner, resource_public):
    if is_admin or is_owner or resource_public:
        return True
    return False

def safe_divide(x, y, check_positive):
    if y != 0 and (not check_positive or x > 0):
        return x / y
    return None

def nested(a, b, c, d):
    if (a and b) or (c and d):
        return 1
    return 0
'''

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(sample)
    tmp = f.name

r = subprocess.run(['python', '-m', 'pymcdc', tmp], capture_output=True, text=True, timeout=30)
print('STDOUT:')
print(r.stdout)
print('STDERR:')
print(r.stderr)
print('RETURNCODE:', r.returncode)
os.unlink(tmp)

STDOUT:
Line number: (3, 7)
Decision: is_admin or is_owner or resource_public
Combinations to be covered: 
    | Result.   is_admin    is_owner    resource_public   Cover. 
-----------------------------------------------------------------
  1 |  False     False       False           False        False  
  2 |   True      True        ----           ----         False  
  3 |   True     False        True           ----         False  
  4 |   True     False       False           True         False  

Line number: (8, 7)
Decision: y != 0 and (not check_positive or x > 0)
Combinations to be covered: 
    | Result.   y != 0    check_positive    x > 0   Cover. 
-----------------------------------------------------------
  1 |  False    False         False         ----    False  
  2 |   True     True         False         ----    False  
  3 |  False     True          True         False   False  
  4 |   True     True          True         True    False  

Line number: (13, 7)
Decision: (a a

In [18]:
# Raw pymcdc output on redditwarp
import subprocess, os
target = os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'tests', 'core', 'test_authorizer_ASYNC.py')
r = subprocess.run(['python', '-m', 'pymcdc', target], capture_output=True, text=True, timeout=60)
print('STDOUT:')
print(r.stdout)
print('STDERR:')
print(r.stderr)
print('RETURNCODE:', r.returncode)

STDOUT:

Covered 0 out of 0 requirements in 0 decisions (100%)
	      
Run time: 0.00432 

STDERR:

RETURNCODE: 0


In [5]:
# Manual raw MC/DC decision extraction using AST
import ast, json

source = open(os.path.join(os.path.dirname(os.getcwd()), 'redditwarp', 'redditwarp', 'spaces', 'discrete.py')).read()
tree = ast.parse(source)

decisions = []
for node in ast.walk(tree):
    if isinstance(node, (ast.If, ast.While, ast.Assert)):
        raw = ast.dump(node.test)
        # count atomic conditions (Name/Compare nodes)
        conditions = [n for n in ast.walk(node.test) if isinstance(n, (ast.Compare, ast.Name, ast.BoolOp))]
        decisions.append({
            'node_type': type(node).__name__,
            'lineno': node.lineno,
            'test_ast': raw,
            'condition_node_count': len(conditions)
        })

print(json.dumps(decisions, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'f:\\testable-whitebox-metrics\\redditwarp\\redditwarp\\spaces\\discrete.py'